# Jour 1 · Pourquoi analyser des séries temporelles IoT ?


## Objectifs

- comprendre la mission globale des trois journées
- relier prévision, anomalie et maintenance prédictive
- situer Pandas, Prophet, Isolation Forest et l'autoencoder dans une même chaîne

## Notre mission

Une installation HVAC envoie toutes les quinze minutes des mesures de température, d'humidité, de puissance électrique, de pression et de vibration.

Pendant trois jours, nous allons chercher à répondre à quatre questions :

1. **Que s'est-il passé ?** — rendre les données fiables et comprendre le comportement observé ;
2. **Que devrait-il se passer ensuite ?** — prévoir un comportement attendu ;
3. **Ce qui se passe est-il inhabituel ?** — produire un score ou une alerte ;
4. **Que doit-on faire ?** — donner du contexte à une décision de maintenance.

![Installation HVAC équipée de capteurs, courbes temporelles et technicien de maintenance](../assets/jour_01/00_hvac_capteurs_maintenance.png)

*Le fil rouge part d'une machine réelle : les capteurs produisent des mesures datées, les courbes rendent le comportement visible et le technicien apporte le contexte terrain.*

## Comment travailler avec ces notebooks ?

Un notebook mélange trois éléments :

- les cellules **Markdown**, qui expliquent une notion et donnent les consignes ;
- les cellules **Code**, que l'on exécute avec `Maj + Entrée` ;
- le **résultat**, affiché juste sous la cellule qui l'a produit.

Lisez toujours l'explication et formulez une attente avant d'exécuter le code : *qu'est-ce que cette cellule devrait afficher ou transformer ?* Après l'exécution, comparez le résultat à cette attente.

Il n'est pas nécessaire de mémoriser toute la syntaxe Pandas. L'objectif est de savoir :

1. quel problème de données on cherche à résoudre ;
2. quelle opération convient ;
3. comment vérifier qu'elle a produit le résultat attendu ;
4. quel risque elle peut introduire.

En cas d'erreur, lisez d'abord la dernière ligne du message : elle indique généralement le type du problème. Vérifiez ensuite que les cellules précédentes ont bien été exécutées dans l'ordre.

## Qu'est-ce qu'une série temporelle ?

Une série temporelle est une suite de mesures associées à des instants et **ordonnées dans le temps**.

| Exemple | Valeur | Pourquoi le temps compte |
|---|---|---|
| Thermostat | température | le cycle jour/nuit se répète |
| Compteur électrique | puissance | la consommation dépend de l'activité |
| Pompe industrielle | vibration | une dérive peut annoncer une usure |
| Serveur | CPU et mémoire | une hausse persistante peut précéder une saturation |

Dans un tableau classique, mélanger les lignes ne change parfois rien. Dans une série temporelle, mélanger les lignes détruit l'histoire : on ne sait plus ce qui vient avant ou après.

## Le fil rouge des trois jours

```text
Capteurs IoT
    ↓
Collecter et horodater
    ↓
Nettoyer et remettre à fréquence régulière
    ↓
Visualiser et comprendre les motifs
    ↓
Prévoir le comportement attendu
    ↓
Mesurer l'écart entre attendu et observé
    ↓
Détecter et prioriser les anomalies
    ↓
Aider une décision de maintenance
```

Chaque étape dépend de la précédente. Un modèle sophistiqué ne répare pas des timestamps incohérents ou des mesures manquantes invisibles.

![Chaîne allant des capteurs à une décision de maintenance](../assets/jour_01/00_chaine_iot_maintenance.png)

*La donnée ne devient utile qu'en traversant une chaîne complète, jusqu'à une décision humaine vérifiable.*

## La carte du parcours

| Journée | Question principale | Notions et outils |
|---|---|---|
| **Jour 1** | Que racontent les données ? | timestamps, qualité, resampling, graphiques, tendance, saisonnalité, moyennes mobiles |
| **Jour 2** | Que devrait-il se passer ensuite ? | baselines, MAE, RMSE, Prophet, résidus de prévision |
| **Jour 3** | Le comportement est-il inhabituel ? | seuils, Z-score, Isolation Forest, autoencoder, alertes de maintenance |

Nous irons toujours du plus simple au plus complexe. Une méthode complexe ne sera conservée que si elle apporte quelque chose de mesurable ou d'utile.

## À quoi serviront les différents outils ?

- **Pandas** : charger, dater, trier, nettoyer et agréger les mesures.
- **Moyenne mobile** : rendre le niveau local et les évolutions lentes plus visibles.
- **Baseline** : disposer d'une prévision simple à battre.
- **MAE et RMSE** : quantifier les erreurs de prévision.
- **Prophet** : modéliser simplement tendance et motifs saisonniers pour prévoir le futur.
- **Seuil métier** : traduire une limite physique ou opérationnelle connue.
- **Z-score mobile** : repérer un écart important par rapport au passé récent.
- **Isolation Forest** : chercher des combinaisons de mesures inhabituelles.
- **Autoencoder** : apprendre à reconstruire le comportement normal et utiliser l'erreur de reconstruction comme score d'anomalie.

Aucun de ces outils ne connaît à lui seul la cause d'une panne. Ils fournissent des informations qui devront être interprétées.

## Première observation, sans chercher à coder

Exécutez la cellule suivante uniquement pour afficher une semaine de mesures. Il n'est pas encore nécessaire de comprendre chaque instruction Python : nous les apprendrons progressivement.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
overview = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_clean.csv")
overview["timestamp"] = pd.to_datetime(overview["timestamp"], utc=True)
overview = overview.set_index("timestamp").sort_index()
overview = overview[["temperature_c", "power_kw", "vibration_mm_s"]].resample("1h").mean()
observation_window = overview.loc["2026-07-29":"2026-08-05"]

axes = observation_window.plot(
    subplots=True,
    figsize=(13, 8),
    sharex=True,
    title=["Température", "Puissance électrique", "Vibration"],
)
axes[0].set_ylabel("°C")
axes[1].set_ylabel("kW")
axes[2].set_ylabel("mm/s")
plt.suptitle("Une semaine dans la vie de hvac_01", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## Une anomalie n'est pas encore un diagnostic

Une alerte peut correspondre à :

- un défaut réel de l'équipement ;
- un capteur défaillant ou mal calibré ;
- un changement normal du procédé ;
- une maintenance déjà planifiée ;
- une erreur de collecte ou de timestamp.

Notre système doit donc fournir un **score, du contexte et une priorité**, puis permettre une confirmation humaine. La maintenance prédictive ne consiste pas à remplacer le technicien, mais à l'aider à intervenir au bon moment.

## Ce qu'il y a autour du modèle

Un modèle n'est jamais isolé. Il reçoit des données préparées, produit un score utilisé par un service d'alertes et dépend ensuite du retour du technicien.

![Architecture IoT complète autour du modèle de détection](../assets/jour_01/00_architecture_autour_modele.png)

*La chaîne va du capteur au technicien, puis le retour terrain revient améliorer les règles et les modèles. La surveillance, le versionnement et l'historique rendent le système exploitable.*

Dans une architecture réelle, il faut également surveiller l'absence de données, versionner le modèle et ses paramètres, conserver l'historique des alertes et recueillir le retour du terrain.

## Ce que nous ne chercherons pas à maîtriser ici

Nous ne ferons pas de démonstrations mathématiques avancées, d'ARIMA détaillé, de CNN, de RNN/LSTM ou d'optimisation poussée. Ces sujets existent, mais notre objectif est d'abord de construire une chaîne IoT/ML simple, compréhensible et fonctionnelle.

À la fin des trois jours, vous devrez surtout pouvoir expliquer :

- pourquoi la qualité temporelle vient avant le modèle ;
- comment établir un comportement attendu ;
- comment transformer un écart en alerte ;
- pourquoi une alerte doit rester reliée à une décision métier.

## Vocabulaire de départ

| Terme | Sens simple |
|---|---|
| Timestamp | instant associé à une mesure |
| Fréquence | intervalle habituel entre deux mesures |
| Tendance | évolution lente du niveau général |
| Saisonnalité | motif qui se répète régulièrement |
| Prévision | estimation d'une valeur future |
| Résidu | différence entre valeur observée et valeur prévue |
| Anomalie | comportement suffisamment inhabituel pour être examiné |
| Faux positif | alerte déclenchée sans anomalie confirmée |
| Faux négatif | anomalie présente mais non détectée |

Nous pouvons maintenant ouvrir le CSV brut : cette fois, chaque opération de nettoyage aura une raison dans la chaîne globale.